# 03 · Open LLMs: zero-shot and few-shot prompting
**Owner: Arpita Majumder** (each teammate runs this same notebook with a different `MODEL_ID` on their own Kaggle account, so the GPU quota is shared)

No training. Each article is put into a prompt, and instead of letting the model write free text we read its probability of answering **"Real"** vs **"Fake"** as the next word. That gives a score for every article (so we can also report AUC) and no answers that fail to parse.

- **zero_shot**: instruction + article
- **few_shot_k4**: 2 real + 2 fake training examples first, repeated with 3 different example sets (seeds 0, 1, 2) for mean ± std

Evaluated on the shared subsets: 1,000 BanFakeNews-2.0 test articles, the external test set, and 1,000 BanglaFakeNews2025 articles. BanglaBERT and the SVM are scored on exactly the same articles.

Rough time on a Kaggle T4 with a 7–9B model in 4-bit: 3–5 hours for everything. Check the speed printed in the first minutes; if it is too slow, lower `QUERY_CHARS` or use one fewer few-shot seed.

**Kaggle settings for this notebook** (right-hand panel → *Session options*):
- **Accelerator:** GPU T4 x2
- **Internet:** On (needed to download the model)
- **Inputs** (*Add Input*): `hrithikmajumdar/bangla-fake-news` (BanFakeNews-2.0) and your team's `banglafakenews2025` dataset. If you also have `New_Test_Dataset.csv` (the 1,000-article external test set of BanFakeNews-2.0), add it too; it is detected automatically.

Label convention everywhere: **1 = Real, 0 = Fake**. Run once with `FAST_DEV_RUN = True` (a few minutes) to check everything works, then set it to `False` and use **Save Version → Save & Run All** so it runs in the background.

**For Llama and Gemma** (gated models): accept the licence on the model's Hugging Face page, create a read token at huggingface.co/settings/tokens, and add it in Kaggle under *Add-ons → Secrets* with the name `HF_TOKEN`. Qwen needs no token.

In [1]:
!pip install -q -U bitsandbytes accelerate

In [2]:
# ======================================================================
# COMMON DATA CODE  (identical in all 4 notebooks; if you change it,
# change it in every notebook, otherwise the example ids will not match)
# ======================================================================
import os, re, glob, json, random, hashlib, unicodedata, warnings
import numpy as np
import pandas as pd

SEED = 42
FAKE, REAL = 0, 1   # label convention everywhere (same as BanFakeNews-2.0 on Kaggle and BanglaFakeNews2025)

INPUT_DIR = os.environ.get('BFN_INPUT_DIR', '/kaggle/input')
OUT_DIR   = os.environ.get('BFN_OUT_DIR', '/kaggle/working')
PRED_DIR  = os.path.join(OUT_DIR, 'preds')
os.makedirs(PRED_DIR, exist_ok=True)

# Leave as None to auto-detect files under INPUT_DIR. Set a path (or a list of
# paths for bfn2025) only if auto-detection picks the wrong file.
PATHS = dict(
    bfn2_train=None, bfn2_val=None, bfn2_test=None,   # BanFakeNews-2.0 split files
    bfn2_all=None,                                    # ...or one single BanFakeNews-2.0 file
    bfn2_external=None,                               # New_Test_Dataset.csv (460 fake + 540 real)
    bfn2025=None,                                     # BanglaFakeNews2025 Bangla csv file(s)
)

LLM_SUBSET_SIZE  = 1000   # stratified sample of the BanFakeNews-2.0 test split used for ALL models
BFN2025_LLM_SIZE = 1000   # stratified sample of BanglaFakeNews2025 used for ALL models


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_URL = re.compile(r'https?://\S+|www\.\S+')
_HTML = re.compile(r'<[^>]+>')
_WS = re.compile(r'\s+')

def clean_text(s):
    if not isinstance(s, str):
        s = '' if pd.isna(s) else str(s)
    s = unicodedata.normalize('NFC', s)
    s = _HTML.sub(' ', s)
    s = _URL.sub(' ', s)
    s = s.replace('\xa0', ' ')
    return _WS.sub(' ', s).strip()


_COLMAP = {'headline': 'Headline', 'title': 'Headline', 'heading': 'Headline',
           'content': 'Content', 'text': 'Content', 'article': 'Content', 'body': 'Content', 'news': 'Content',
           'label': 'Label', 'class': 'Label', 'target': 'Label',
           'category': 'Category', 'domain': 'Domain',
           'f-type': 'FType', 'f_type': 'FType', 'ftype': 'FType', 'fake_type': 'FType'}

def _std_cols(df, source):
    ren = {}
    for c in df.columns:
        k = str(c).strip().lower()
        if k in _COLMAP and _COLMAP[k] not in ren.values():
            ren[c] = _COLMAP[k]
    df = df.rename(columns=ren)
    missing = {'Content', 'Label'} - set(df.columns)
    if missing:
        raise ValueError(f'[{source}] columns {missing} not found. Columns are {list(df.columns)}. '
                         'Rename them or add them to _COLMAP.')
    if 'Headline' not in df.columns:
        df['Headline'] = ''
    return df


def _to_binary(df, source):
    lab = df['Label']
    if not pd.api.types.is_numeric_dtype(lab):
        low = lab.astype(str).str.strip().str.lower()
        m = {'real': REAL, 'authentic': REAL, 'true': REAL, '1': REAL,
             'fake': FAKE, 'false': FAKE, '0': FAKE}
        bad = set(low.unique()) - set(m)
        if bad:
            raise ValueError(f'[{source}] unknown label values {bad}')
        df['Label'] = low.map(m).astype(int)
        return df
    df = df[lab.notna()].copy()
    lab = df['Label'].astype(int)
    vals = set(lab.unique())
    if vals <= {0, 1}:
        df['Label'] = lab
        return df
    if vals == {0, 1, 2, 3}:
        # GitHub release of BanFakeNews-2.0: 3 = authentic, 0/1/2 = fake sub-types
        print(f'[{source}] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)')
        df['FakeType'] = np.where(lab == 3, 'Not Applicable', 'type_' + lab.astype(str))
        df['Label'] = (lab == 3).astype(int)
        return df
    raise ValueError(f'[{source}] unexpected label values {vals}')


def _read(path):
    if path.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    return pd.read_csv(path, encoding='utf-8-sig')


def _bangla_ratio(texts):
    s = ''.join(texts)[:50000]
    letters = [ch for ch in s if ch.isalpha() or 'ঀ' <= ch <= '৿']
    return sum('ঀ' <= ch <= '৿' for ch in letters) / max(1, len(letters))


def _prepare(df, source):
    df = _std_cols(df.copy(), source)
    df = _to_binary(df, source)
    df['Headline'] = df['Headline'].map(clean_text)
    df['Content'] = df['Content'].map(clean_text)
    n0 = len(df)
    df = df[(df['Headline'].str.len() + df['Content'].str.len()) >= 20].copy()
    df['key'] = (df['Headline'] + '||' + df['Content']).map(lambda s: hashlib.md5(s.encode('utf-8')).hexdigest())
    df = df.drop_duplicates('key')
    if n0 - len(df):
        print(f'[{source}] dropped {n0 - len(df)} empty or duplicate rows')
    for col in ['Category', 'FType']:   # tidy labels like 'Crime ', 'Miscellaneous\t', 'Entertainmen'
        if col in df.columns:
            v = df[col].astype(str).str.strip()
            names = set(v.unique())
            fix = {}
            for a in names:
                longer = [b for b in names if b != a and b.startswith(a) and len(b) - len(a) <= 2]
                if len(longer) == 1:
                    fix[a] = longer[0]
            df[col] = v.replace(fix)
    keep = [c for c in ['Headline', 'Content', 'Label', 'Category', 'Domain', 'FType', 'FakeType', 'key']
            if c in df.columns]
    return df[keep].reset_index(drop=True)


def _all_files():
    fs = []
    for ext in ('csv', 'xlsx', 'xls'):
        fs += glob.glob(os.path.join(INPUT_DIR, '**', f'*.{ext}'), recursive=True)
    # never treat our own prediction files as data
    return sorted(f for f in fs if os.sep + 'preds' + os.sep not in f)


def _locate():
    found = {k: v for k, v in PATHS.items() if v}
    if isinstance(found.get('bfn2025'), str):
        found['bfn2025'] = [found['bfn2025']]
    files = _all_files()
    base = lambda f: os.path.basename(f).lower()
    if 'bfn2_external' not in found:
        c = [f for f in files if re.search(r'new[ _-]?test', base(f))]
        if c:
            found['bfn2_external'] = c[0]
    if 'bfn2025' not in found:
        c = [f for f in files if '2025' in f.lower()]
        if c:
            found['bfn2025'] = c
    used = {found.get('bfn2_external')} | set(found.get('bfn2025') or [])
    rest = [f for f in files if f not in used]
    for split, keys in [('bfn2_train', ['train']), ('bfn2_val', ['val', 'valid', 'dev']), ('bfn2_test', ['test'])]:
        if split not in found:
            c = [f for f in rest if any(k in base(f) for k in keys)]
            if c:
                found[split] = c[0]
    have_splits = all(k in found for k in ['bfn2_train', 'bfn2_val', 'bfn2_test'])
    if not have_splits and 'bfn2_all' not in found:
        c = [f for f in rest if f not in found.values()]
        if c:
            found['bfn2_all'] = max(c, key=os.path.getsize)   # biggest remaining file
    return found


def _strat_flag(df, n):
    from sklearn.model_selection import train_test_split
    if n is None or n >= len(df):
        return np.ones(len(df), dtype=bool)
    idx, _ = train_test_split(np.arange(len(df)), train_size=n, stratify=df['Label'], random_state=SEED)
    flag = np.zeros(len(df), dtype=bool)
    flag[idx] = True
    return flag


def load_all():
    """Returns dict: train, val, bfn2_test, and (if found) bfn2_external, bfn2025."""
    from sklearn.model_selection import train_test_split
    f = _locate()
    print('Files used:')
    for k, v in f.items():
        print(f'  {k}: {v}')
    if all(k in f for k in ['bfn2_train', 'bfn2_val', 'bfn2_test']):
        tr, va, te = (_prepare(_read(f[k]), k) for k in ['bfn2_train', 'bfn2_val', 'bfn2_test'])
    elif 'bfn2_all' in f:
        alldf = _prepare(_read(f['bfn2_all']), 'bfn2_all')
        tr, tmp = train_test_split(alldf, test_size=0.30, stratify=alldf['Label'], random_state=SEED)
        va, te = train_test_split(tmp, test_size=0.50, stratify=tmp['Label'], random_state=SEED)
        print('One BanFakeNews-2.0 file found -> stratified 70/15/15 split with seed 42 (same ratio as the paper)')
    else:
        raise FileNotFoundError(f'BanFakeNews-2.0 not found under {INPUT_DIR}. Add it with "Add Input".')

    # no article may appear in both training data and any evaluation set
    seen = set(tr['key'])
    va = va[~va['key'].isin(seen)]
    te = te[~te['key'].isin(seen | set(va['key']))]
    data = {'train': tr, 'val': va, 'bfn2_test': te}

    if 'bfn2_external' in f:
        data['bfn2_external'] = _prepare(_read(f['bfn2_external']), 'bfn2_external')
    if 'bfn2025' in f:
        parts = []
        for p in f['bfn2025']:
            d = _std_cols(_read(p), p)
            r = _bangla_ratio(d['Content'].astype(str).head(200).tolist())
            if r < 0.5:
                print(f'  skipping {os.path.basename(p)} (not Bangla text, Bangla ratio {r:.2f})')
                continue
            parts.append(d)
        if parts:
            data['bfn2025'] = _prepare(pd.concat(parts, ignore_index=True), 'bfn2025')

    for name in ['bfn2_external', 'bfn2025']:
        if name in data:
            n0 = len(data[name])
            data[name] = data[name][~data[name]['key'].isin(seen)]
            if n0 - len(data[name]):
                print(f'[{name}] removed {n0 - len(data[name])} articles that also appear in the training split')

    for name, df in data.items():
        df = df.reset_index(drop=True)
        df.insert(0, 'id', [f'{name}_{i:05d}' for i in range(len(df))])
        data[name] = df
    data['bfn2_test']['in_llm_subset'] = _strat_flag(data['bfn2_test'], LLM_SUBSET_SIZE)
    if 'bfn2_external' in data:
        data['bfn2_external']['in_llm_subset'] = True
    if 'bfn2025' in data:
        data['bfn2025']['in_llm_subset'] = _strat_flag(data['bfn2025'], BFN2025_LLM_SIZE)

    print('\nSplit sizes (fake / total):')
    for name, df in data.items():
        print(f'  {name:14s} {int((df.Label == FAKE).sum()):6d} / {len(df):6d}')
    for k in ['bfn2_external', 'bfn2025']:
        if k not in data:
            print(f'  NOTE: {k} not found -> it will be skipped. Add it as an input to include it.')
    return data


def eval_sets(data, llm_only=False):
    """Evaluation sets. The *_llm sets are the shared subsets every model is compared on."""
    te = data['bfn2_test']
    out = {} if llm_only else {'bfn2_test': te}
    out['bfn2_test_llm'] = te[te['in_llm_subset']]
    if 'bfn2_external' in data:
        out['bfn2_external'] = data['bfn2_external']
    if 'bfn2025' in data:
        d = data['bfn2025']
        if not llm_only:
            out['bfn2025'] = d
        out['bfn2025_llm'] = d[d['in_llm_subset']]
    return {k: v.reset_index(drop=True) for k, v in out.items()}


def compute_metrics(y_true, y_pred, p_fake=None):
    from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                                 matthews_corrcoef, roc_auc_score, average_precision_score)
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    m = dict(n=int(len(y_true)), n_fake=int((y_true == FAKE).sum()),
             macro_f1=f1_score(y_true, y_pred, average='macro', labels=[FAKE, REAL], zero_division=0),
             fake_f1=f1_score(y_true, y_pred, pos_label=FAKE, zero_division=0),
             fake_precision=precision_score(y_true, y_pred, pos_label=FAKE, zero_division=0),
             fake_recall=recall_score(y_true, y_pred, pos_label=FAKE, zero_division=0),
             real_f1=f1_score(y_true, y_pred, pos_label=REAL, zero_division=0),
             accuracy=accuracy_score(y_true, y_pred),
             mcc=matthews_corrcoef(y_true, y_pred),
             pred_fake_rate=float((y_pred == FAKE).mean()))
    if p_fake is not None and len(set(y_true)) == 2:
        is_fake = (y_true == FAKE).astype(int)
        m['roc_auc'] = roc_auc_score(is_fake, p_fake)
        m['pr_auc_fake'] = average_precision_score(is_fake, p_fake)
    return {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in m.items()}


def _slug(s):
    return re.sub(r'[^A-Za-z0-9.+\-]+', '-', s).strip('-')


def report(model, setting, seed, set_name, df, y_pred, p_fake=None, extra=None):
    """Prints metrics, appends them to results.jsonl and saves per-article predictions."""
    m = compute_metrics(df['Label'].values, y_pred, p_fake)
    rec = dict(model=model, setting=setting, seed=int(seed), eval_set=set_name, **m, **(extra or {}))
    with open(os.path.join(OUT_DIR, 'results.jsonl'), 'a', encoding='utf-8') as fh:
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n')
    out = pd.DataFrame({'id': df['id'].values, 'y_true': df['Label'].values,
                        'y_pred': np.asarray(y_pred).astype(int)})
    if p_fake is not None:
        out['p_fake'] = np.round(np.asarray(p_fake, dtype=float), 5)
    out.to_csv(os.path.join(PRED_DIR, f'{_slug(model)}__{setting}__{set_name}__seed{seed}.csv'), index=False)
    auc = f"  AUC={m['roc_auc']:.4f}" if 'roc_auc' in m else ''
    print(f"{model:24s} {setting:12s} seed={seed:<3d} {set_name:14s} n={m['n']:5d}  "
          f"macroF1={m['macro_f1']:.4f}  fakeF1={m['fake_f1']:.4f}{auc}")
    return rec

In [3]:
import gc, time, inspect
import torch
import transformers
from packaging import version
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ----- choose ONE model per run -----
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
#   'meta-llama/Llama-3.1-8B-Instruct'   (gated: needs HF_TOKEN)
#   'google/gemma-2-9b-it'               (gated: needs HF_TOKEN)

SETTINGS      = ['zero_shot', 'few_shot']
K_SHOT        = 4            # 2 real + 2 fake examples
FEWSHOT_SEEDS = [0, 1, 2]    # 3 different example sets -> mean +- std
QUERY_CHARS   = 800          # characters of the article content shown to the model
DEMO_CHARS    = 300          # characters of each example's content
BATCH_SIZE    = 4            # halves automatically on out-of-memory
LOAD_IN_4BIT  = True
FAST_DEV_RUN  = False        # True = 40 articles per set, 1 few-shot seed

print('transformers', transformers.__version__, '| GPUs:', torch.cuda.device_count())

transformers 5.16.1 | GPUs: 2


In [4]:
def hf_login():
    tok = os.environ.get('HF_TOKEN')
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            tok = None
    if tok:
        from huggingface_hub import login
        login(token=tok); print('Logged in to Hugging Face')
    else:
        print('No HF_TOKEN found (fine for Qwen; Llama and Gemma need it)')

def load_llm(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'                 # so the last position is always the answer slot
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(device_map='auto')
    if LOAD_IN_4BIT and torch.cuda.is_available():
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16,
                                                       bnb_4bit_use_double_quant=True)
    if 'gemma' in model_id.lower():
        kw['attn_implementation'] = 'eager'   # recommended for Gemma-2
    dtype_key = 'dtype' if version.parse(transformers.__version__) >= version.parse('4.56') else 'torch_dtype'
    kw[dtype_key] = torch.float16             # T4/P100 have no bfloat16
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    model.eval()
    return tok, model

hf_login()
tok, model = load_llm(MODEL_ID)
SHORT = MODEL_ID.split('/')[-1]
FIRST_DEVICE = model.get_input_embeddings().weight.device
_fwd = inspect.signature(model.forward).parameters
LOGITS_KW = 'logits_to_keep' if 'logits_to_keep' in _fwd else ('num_logits_to_keep' if 'num_logits_to_keep' in _fwd else None)
print(SHORT, 'loaded on', FIRST_DEVICE, '| last-token-only logits via', LOGITS_KW)

No HF_TOKEN found (fine for Qwen; Llama and Gemma need it)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B-Instruct loaded on cuda:0 | last-token-only logits via logits_to_keep


## 1. Prompt and answer scoring

In [5]:
INSTRUCTION = ("You are a fact-checking assistant for Bangla (Bengali) news. Read the news article below and decide "
               "whether it is Real (authentic news from a credible outlet) or Fake (fabricated, misleading, satire "
               "or clickbait). Answer with exactly one word: Real or Fake.")

def article_block(row, max_chars):
    c = row.Content[:max_chars] + ('…' if len(row.Content) > max_chars else '')
    return f'Headline: {row.Headline}\nContent: {c}\n\nAnswer:'

def build_messages(row, demos=None):
    msgs = []
    for i, d in enumerate(demos or []):
        prefix = INSTRUCTION + '\n\n' if i == 0 else ''
        msgs.append({'role': 'user', 'content': prefix + article_block(d, DEMO_CHARS)})
        msgs.append({'role': 'assistant', 'content': 'Real' if d.Label == REAL else 'Fake'})
    prefix = '' if demos else INSTRUCTION + '\n\n'
    msgs.append({'role': 'user', 'content': prefix + article_block(row, QUERY_CHARS)})
    return msgs

def render(msgs):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def answer_token_ids():
    """First token of each spelling of the answer. A token is kept only if it is the start of the word
    (not a bare space); single-letter tokens are dropped when a longer one exists."""
    ids = {}
    for lab, word in [(REAL, 'real'), (FAKE, 'fake')]:
        cand = {}
        for w in [word.capitalize(), ' ' + word.capitalize(), word, ' ' + word]:
            t = tok.encode(w, add_special_tokens=False)
            piece = tok.decode(t[:1]).strip().lower() if t else ''
            if piece and word.startswith(piece):
                cand[t[0]] = len(piece)
        assert cand, f'no usable first token for {word!r}'
        longest = max(cand.values())
        ids[lab] = sorted(i for i, n in cand.items() if n > 1 or longest == 1)
    assert not set(ids[REAL]) & set(ids[FAKE]), f'Real/Fake share a first token: {ids}'
    print('answer tokens  Real:', [tok.decode([i]) for i in ids[REAL]], ' Fake:', [tok.decode([i]) for i in ids[FAKE]])
    return ids

ANS = answer_token_ids()

@torch.no_grad()
def score(texts, batch_size=BATCH_SIZE, label=''):
    """Returns P(Fake) per prompt and the probability mass on the two answers (should be close to 1)."""
    order = np.argsort([-len(t) for t in texts])   # longest first: any out-of-memory happens early
    p_fake = np.zeros(len(texts)); mass = np.zeros(len(texts))
    i, bs, t0 = 0, batch_size, time.time()
    while i < len(order):
        idx = order[i:i + bs]
        try:
            enc = tok([texts[j] for j in idx], return_tensors='pt', padding=True, add_special_tokens=False).to(FIRST_DEVICE)
            pos = (enc.attention_mask.cumsum(-1) - 1).clamp(min=0)
            kw = {LOGITS_KW: 1} if LOGITS_KW else {}
            logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                           position_ids=pos, use_cache=False, **kw).logits[:, -1, :].float()
        except torch.cuda.OutOfMemoryError:
            enc = logits = None; gc.collect(); torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2); print(f'  out of memory -> batch size {bs}'); continue
        logp = torch.log_softmax(logits, -1)
        lr = torch.logsumexp(logp[:, ANS[REAL]], -1)
        lf = torch.logsumexp(logp[:, ANS[FAKE]], -1)
        p_fake[idx] = torch.sigmoid(lf - lr).cpu().numpy()
        mass[idx] = (lr.exp() + lf.exp()).cpu().numpy()
        i += len(idx)
        if i == len(idx) or i % 200 < len(idx) or i == len(order):
            el = time.time() - t0
            print(f'  {label} {i}/{len(order)}  {el / 60:.1f} min elapsed, ~{el / i * (len(order) - i) / 60:.1f} min left')
    return p_fake, mass

def pick_demos(train, k, seed):
    pool = train[(train['Content'].str.len() >= 200) & (train['Headline'].str.len() > 0)]
    real = pool[pool.Label == REAL].sample(k // 2, random_state=seed)
    fake = pool[pool.Label == FAKE].sample(k - k // 2, random_state=seed + 1000)
    return list(pd.concat([real, fake]).sample(frac=1, random_state=seed).itertuples(index=False))

answer tokens  Real: [' real', 'real', ' Real', 'Real']  Fake: [' fake', 'fake', ' Fake', 'Fake']


## 2. Data, and a look at one prompt

In [6]:
data = load_all()
sets = eval_sets(data, llm_only=True)
if FAST_DEV_RUN:
    sets = {k: pd.concat([v[v.Label == l].head(20) for l in (REAL, FAKE)]).reset_index(drop=True) for k, v in sets.items()}
    FEWSHOT_SEEDS = FEWSHOT_SEEDS[:1]

example = render(build_messages(next(sets['bfn2_test_llm'].itertuples(index=False)), pick_demos(data['train'], K_SHOT, 0)))
print(example[:3000])
n_tok = [len(tok(render(build_messages(r, pick_demos(data['train'], K_SHOT, 0))), add_special_tokens=False).input_ids)
         for r in sets['bfn2_test_llm'].head(20).itertuples(index=False)]
print('\nfew-shot prompt length in tokens: median', int(np.median(n_tok)), 'max', max(n_tok))

Files used:
  bfn2025: ['/kaggle/input/datasets/mahbubulalamabid/banglafakenews2025/Bangla_Fake_News_Dataset.xlsx', '/kaggle/input/datasets/mahbubulalamabid/banglafakenews2025/Bangla_Real_News_Dataset.xlsx']
  bfn2_train: /kaggle/input/datasets/hrithikmajumdar/bangla-fake-news/train_cleaned.csv
  bfn2_val: /kaggle/input/datasets/hrithikmajumdar/bangla-fake-news/val_cleaned.csv
  bfn2_test: /kaggle/input/datasets/hrithikmajumdar/bangla-fake-news/test_cleaned.csv
[bfn2_train] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)
[bfn2_train] dropped 1510 empty or duplicate rows
[bfn2_val] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)
[bfn2_val] dropped 75 empty or duplicate rows
[bfn2_test] 4-class labels found -> 3 = Real, 0/1/2 = Fake (sub-type kept in column FakeType)
[bfn2_test] dropped 83 empty or duplicate rows
[bfn2025] dropped 8 empty or duplicate rows

Split sizes (fake / total):
  train            7243 /  40870
  v

## 3. Run zero-shot and few-shot
`answer mass` is how much probability the model puts on Real/Fake at all. If it is below about 0.5 the model wants to say something else first; check the prompt before trusting the numbers.

In [7]:
runs = []
if 'zero_shot' in SETTINGS:
    runs.append(('zero_shot', 0, None))
if 'few_shot' in SETTINGS:
    runs += [(f'few_shot_k{K_SHOT}', s, pick_demos(data['train'], K_SHOT, s)) for s in FEWSHOT_SEEDS]

for setting, seed, demos in runs:
    for name, df in sets.items():
        texts = [render(build_messages(r, demos)) for r in df.itertuples(index=False)]
        p, mass = score(texts, label=f'{setting} s{seed} {name}')
        report(SHORT, setting, seed, name, df, np.where(p >= 0.5, FAKE, REAL), p,
               extra={'answer_mass': round(float(mass.mean()), 4), 'query_chars': QUERY_CHARS})
        print(f'   answer mass: mean {mass.mean():.3f}, min {mass.min():.3f}')

  zero_shot s0 bfn2_test_llm 4/1000  0.1 min elapsed, ~17.7 min left
  zero_shot s0 bfn2_test_llm 200/1000  3.0 min elapsed, ~11.9 min left
  zero_shot s0 bfn2_test_llm 400/1000  6.0 min elapsed, ~8.9 min left
  zero_shot s0 bfn2_test_llm 600/1000  8.9 min elapsed, ~5.9 min left
  zero_shot s0 bfn2_test_llm 800/1000  11.8 min elapsed, ~3.0 min left
  zero_shot s0 bfn2_test_llm 1000/1000  14.4 min elapsed, ~0.0 min left
Qwen2.5-7B-Instruct      zero_shot    seed=0   bfn2_test_llm  n= 1000  macroF1=0.6877  fakeF1=0.4653  AUC=0.7316
   answer mass: mean 1.000, min 0.996
  zero_shot s0 bfn2025_llm 4/1000  0.1 min elapsed, ~15.7 min left
  zero_shot s0 bfn2025_llm 200/1000  3.1 min elapsed, ~12.3 min left
  zero_shot s0 bfn2025_llm 400/1000  6.1 min elapsed, ~9.1 min left
  zero_shot s0 bfn2025_llm 600/1000  9.1 min elapsed, ~6.0 min left
  zero_shot s0 bfn2025_llm 800/1000  11.4 min elapsed, ~2.8 min left
  zero_shot s0 bfn2025_llm 1000/1000  12.5 min elapsed, ~0.0 min left
Qwen2.5-7B-Inst

In [8]:
res = pd.read_json(os.path.join(OUT_DIR, 'results.jsonl'), lines=True)
res = res[res.model == SHORT].drop_duplicates(['setting', 'seed', 'eval_set'], keep='last')
display(res.groupby(['setting', 'eval_set'])[['macro_f1', 'fake_f1', 'roc_auc', 'pred_fake_rate', 'answer_mass']].agg(['mean', 'std']).round(4))

macro_f1         fake_f1         roc_auc          \
                              mean     std    mean     std    mean     std   
setting     eval_set                                                         
few_shot_k4 bfn2025_llm     0.8475  0.0061  0.8501  0.0040  0.9317  0.0063   
            bfn2_test_llm   0.6529  0.0436  0.4585  0.0446  0.7526  0.0059   
zero_shot   bfn2025_llm     0.7536     NaN  0.7110     NaN  0.8853     NaN   
            bfn2_test_llm   0.6877     NaN  0.4653     NaN  0.7316     NaN   

                          pred_fake_rate         answer_mass          
                                    mean     std        mean     std  
setting     eval_set                                                  
few_shot_k4 bfn2025_llm           0.5143  0.0661      0.9997  0.0002  
            bfn2_test_llm         0.2827  0.0736      0.9999  0.0000  
zero_shot   bfn2025_llm           0.3260     NaN      0.9997     NaN  
            bfn2_test_llm         0.1370     NaN      0.9998     NaN